#Handling None

###Droping Nones

In [0]:
from pyspark.sql.functions import *

data=(
    [(1,"abc",None),
     (2,"def","xyz"),
     (3,"ghi","xyz"),
     (None,None,None),
     (5,"mno",None),
     (6,"pqr","pqr"),
     (7,"stu",None),
     (8,"vwx","abc"),
     (9,"yz",None)]
    )

schema=(["id","name","address"])
df_drop=spark.createDataFrame(data,schema)

In [0]:
display(df_drop)

###All Nulls

In [0]:
df_all_null=df_drop.dropna("all")
display(df_all_null)


###Any Null

In [0]:
df_any_null=df_drop.dropna("any")
display(df_any_null)

###Subset

In [0]:
df_subset_null=df_drop.dropna(subset=["name","address"],how="all")

display(df_subset_null)

###Fill nulls

In [0]:
df_fillna=df_drop.fillna("NA")
display(df_fillna)

In [0]:
df_custom_na=df_drop.fillna({"id":0,"name":"NAME_NA","address":"ADDRESS_NA"})
display(df_custom_na)


#Split & Indexing

In [0]:
df_csv=spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv")

display(df_csv)


In [0]:
df_split=(df_csv.withColumn("street_address",split("shipping_address",",")[0])
    .withColumn("city_name",split("shipping_address",",")[1])
)

display(df_split)

#Explode

In [0]:
df_arr=df_split.withColumn("array_address",split("shipping_address",","))\
        .select("order_id","shipping_address","array_address")

display(df_arr)

In [0]:
df_exp=df_arr.withColumn("address_exploded",explode("array_address"))

display(df_exp)

In [0]:
df_exp_outer=df_arr.withColumn("address_exploded",explode_outer("array_address"))

display(df_exp_outer)

#Array Contains

In [0]:
#df_arrary_contains=df_arr.filter(array_contains("array_address"," City1"))
df_array_contains=df_arr.withColumn("city_1_flag",array_contains("array_address"," City1"))
display(df_array_contains)

#Group By

In [0]:
from pyspark.sql.functions import col,sum,round
df_agg=df_csv.withColumn("total_price",col("quantity")*col("price"))\
        .groupBy("product_id").agg(round(sum("total_price"),2).alias("tot_price"))
display(df_agg)

In [0]:
df_agg=df_csv.withColumn("total_price",col("quantity")*col("price"))\
        .groupBy("product_id")\
        .agg(sum("total_price").alias("tot_price"))\
        .withColumn("tot_price",round(col("tot_price"),2))\
        .sort(col("tot_price").desc())

display(df_agg)

In [0]:
from pyspark.sql.functions import col,sum,avg

df_agg_new=df_csv.withColumn("total_price",col("quantity")*col("price"))\
        .groupBy("product_id","customer_id")\
        .agg(sum("total_price").alias("tot_price")\
        ,avg("total_price").alias("avg_price"))\
        .withColumn("tot_price",round(col("tot_price"),2))\
        .withColumn("avg_price",round(col("avg_price"),2))\
        .sort("product_id","tot_price",ascending=[True,False])


display(df_agg_new)

#Approx Count

In [0]:
from pyspark.sql.functions import approx_count_distinct
df=df_csv.groupBy("product_id").agg(approx_count_distinct("customer_id").alias("distinct_customers"))

display(df)

In [0]:
from pyspark.sql.functions import countDistinct
df=df_csv.groupBy("product_id").agg(countDistinct("customer_id").alias("distinct_customers"))

display(df)

#Collect List

In [0]:
from pyspark.sql.functions import collect_list
df_collect=df_csv.groupBy("customer_id")\
        .agg(collect_list("product_id").alias("product_id_list"))

display(df_collect)